<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

# SemEval-2026 Task 3 (Track A: DimABSA, Track B: DimStance)
# Subtask 1: Dimensional Aspect Sentiment Regression (DimASR)

-----

## Starter Notebook
Leveraging Pretrained Language Models for Dimensional Sentiment Regression


## Introduction:

You are welcome to participate in our SemEval Shared Task!

In this starter notebook, we will take you through the process of fine-tuning a pre-trained language model on a sample data to build a sentiment regressor. The notebook was adapted from a Hugginface implementation for such tasks.

### Outline:

- Installation and importation of necessary libraries
Setting up the project parameters.
Running training and evaluation
Before you start:

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:

The codes in this notebook are provided to familiarize yourselves with fine-tuning language models for sentiment regression. You may extend and (or) modify as appropriate to obtain competitive performances.

### Languages and Domains:
#### Track A: Subtask 1
- eng_restaurant
- eng_laptop
- jpn_hotel
- jpn_finance
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop
#### Track B: Subtask 1
- deu-stance
- eng-stance
- hau-stance
- kin-stance
- swa-stance
- twi-stance


### Model:
This Starter Notebook uses the bert-base-multilingual-cased pretrained model, developed by Google. The model was trained with a masked language modeling (MLM) objective on the top 104 languages with the largest Wikipedia presence. You can find the model here: https://huggingface.co/google-bert/bert-base-multilingual-cased

If your target language is not included in the common set supported by this model, you can search for a more suitable model on Hugging Face: https://huggingface.co/models



In [4]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests

# importation des modèles
from baseline_svr import run_svr_baseline


def load_jsonl(filepath: str) -> List[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_jsonl_url(url: str) -> List[Dict]:
    resp = requests.get(url)
    resp.raise_for_status()
    return [json.loads(line) for line in resp.text.splitlines()]

### First, visit the [DimABSA2006](https://github.com/DimABSA/DimABSA2026) repository, check the task-dataset.

### Step 1: Load the competition data

- Read JSONL files (train/dev/predict) into Colab.  
- Train files contain Valence–Arousal (VA) labels.  
- Predict files have no VA labels.  
- This script:
  1. Loads the JSONL data.
  2. Splits 10% of train data as dev set.
  3. Converts JSONL into DataFrames (ID, Text, Aspect, Valence, Arousal).
  4. Prints the first few rows for checking.


In [11]:
#task config
subtask = "subtask_1"#don't change
task = "task1"#don't change
lang = "eng" #chang the language you want to test
domain = "laptop" #change what domain you want to test

train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
predict_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"

#model config
models_dict = [
    {
        "name": "SVR_Baseline",
        "type": "sklearn",
        "max_features": 5000
    },
    {
        "name": "microsoft/deberta-v3-base", # microsoft/mdeberta-v3-base pour la version multilingue
        "type": "transformer",
        "lr": 1e-5,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    },
    {
        "name": "bert-base-multilingual-cased",
        "type": "transformer",
        "lr": 1e-5,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    },
    {
        "name": "roberta-base",
        "type": "transformer",
        "lr": 1e-6,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    }
]

train_raw = load_jsonl_url(train_url)
predict_raw = load_jsonl_url(predict_url)

another transformer models you can try:
1. roberta-large
2. roberta-base
3. bert-base-uncased

more models please visit [huggingface](https://huggingface.co/models)

In [6]:
#==== step 1 load the data ====
# you can change the env for your task.
# train data should have the VA labels, predit data without VA labels

def jsonl_to_df(data):
    if 'Quadruplet' in data[0]:
        df = pd.json_normalize(data, 'Quadruplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Category', 'Opinion'])  # drop unnecessary columns
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')  # remove duplicate ID+Aspect

    elif 'Triplet' in data[0]:
        df = pd.json_normalize(data, 'Triplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Opinion'])  # drop unnecessary columns
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')  # remove duplicate ID+Aspect

    elif 'Aspect_VA' in data[0]:
        df = pd.json_normalize(data, 'Aspect_VA', ['ID', 'Text'])
        df = df.rename(columns={df.columns[0]: "Aspect"})  # rename to Aspect
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop_duplicates(subset=['ID', 'Aspect'], keep='first')  # remove duplicate ID+Aspect

    elif 'Aspect' in data[0]:
        df = pd.json_normalize(data, 'Aspect', ['ID', 'Text'])
        df = df.rename(columns={df.columns[0]: "Aspect"})  # rename to Aspect
        df['Valence'] = 0  # default value
        df['Arousal'] = 0  # default value

    else:
        raise ValueError("Invalid format: must include 'Quadruplet' or 'Triplet' or 'Aspect'")

    return df

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)

### Display the dataframe

In [7]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_laptop train_df

,Aspect,ID,Text,Valence,Arousal
251,computer,laptop_quad_dev_190,"if i had it to do over , i would not purchase ...",3.10,6.30
4516,unit,laptop_quad_train_2141,after charging the unit for 2 hours i discover...,4.75,5.25
335,NULL,laptop_quad_dev_253,"freezes with red lines across it , froze five ...",2.00,7.67
3286,device,laptop_quad_train_1230,a wonderful device with extremely clear display .,8.00,7.83
753,screen,laptop_quad_test_236,the screen does look good .,6.62,6.62


### subtask_1_eng_laptop dev_df

,Aspect,ID,Text,Valence,Arousal
3628,NULL,laptop_quad_train_1485,but it lost the coil whine roulette - - badly .,3.12,6.12
3096,key board,laptop_quad_train_1095,the key board is one of the best i ' ve ever t...,7.67,7.50
4814,sleep time,laptop_quad_train_2357,"- boot time , sleep time and wake time are cra...",7.50,7.50
5443,track pad,laptop_quad_train_2729,please note that the track pad is way better t...,7.12,7.00
197,retina screen,laptop_quad_dev_147,the retina screen is amazing .,8.12,8.25


### subtask_1_eng_laptop predict_df

,Aspect,VA,ID,Text,Valence,Arousal
0,touchscreen,7.80#7.60,lap26_aspect_va_dev_1,The touchscreen works very well,7.80,7.60
1,HP,2.88#6.75,lap26_aspect_va_dev_2,I am so disappointed in HP,2.88,6.75
2,keyboard,6.88#6.62,lap26_aspect_va_dev_3,The keyboard is big enough to use for real typing,6.88,6.62
3,screen size,7.25#7.12,lap26_aspect_va_dev_4,I like the screen size,7.25,7.12
4,Lenovo,7.38#7.38,lap26_aspect_va_dev_5,Lenovo is my favorite brand of computer,7.38,7.38


### Définition des classes et des fonctions


In [8]:
class VADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.sentences = dataframe["Text"].tolist()
        self.aspects = dataframe["Aspect"].tolist()
        self.labels = dataframe[["Valence", "Arousal"]].values.astype(float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        text = f"{self.aspects[idx]}: {self.sentences[idx]}"
        encoded = self.tokenizer(
            text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

class TransformerVARegressor(nn.Module):
    def __init__(self, current_model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(current_model_name)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls_output)
        return self.reg_head(x)

    def train_epoch(self, dataloader, optimizer, loss_fn, device):
        self.train()
        total_loss = 0
        for batch in tqdm(dataloader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = self(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def eval_epoch(self, dataloader, loss_fn, device):
        self.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = self(input_ids, attention_mask)
                loss = loss_fn(outputs, labels)
                total_loss += loss.item()
        return total_loss / len(dataloader)

In [9]:
def get_prd(model,dataloder, type ="dev"):
    if type == "dev":
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].cpu().numpy()
                outputs = model(input_ids, attention_mask).cpu().numpy()
                all_preds.append(outputs)
                all_labels.append(labels)
        preds = np.vstack(all_preds)
        lables = np.vstack(all_labels)

        pred_v = preds[:,0]
        pred_a = preds[:,1]

        gold_v = lables[:,0]
        gold_a = lables[:,1]

        return pred_v, pred_a, gold_v, gold_a

    elif type == "pred":
        all_preds = []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs = model(input_ids, attention_mask).cpu().numpy()
                all_preds.append(outputs)
        preds = np.vstack(all_preds)

        pred_v = preds[:, 0]
        pred_a = preds[:, 1]

        return pred_v, pred_a

def evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v, is_norm=False):
    if not (all(1 <= x <= 9 for x in pred_v) and all(1 <= x <= 9 for x in pred_a)):
        print(f"Warning: Some predicted values are out of the numerical range.")

    # Calcul du PCC (Pearson Correlation Coefficient)
    pcc_v = pearsonr(pred_v, gold_v)[0]
    pcc_a = pearsonr(pred_a, gold_a)[0]

    # Calcul du RMSE séparé pour la Valence et l'Arousal avec Numpy
    rmse_v = np.sqrt(np.mean((gold_v - pred_v)**2))
    rmse_a = np.sqrt(np.mean((gold_a - pred_a)**2))

    # Calcul du RMSE global
    gold_va = np.concatenate((gold_v, gold_a))
    pred_va = np.concatenate((pred_v, pred_a))
    rmse_va_global = np.sqrt(np.mean((gold_va - pred_va)**2))

    # Appliquation de la logique de normalisation si is_norm est True
    if is_norm:
        rmse_v = rmse_v / math.sqrt(128)
        rmse_a = rmse_a / math.sqrt(128)
        rmse_va_global = rmse_va_global / math.sqrt(128)

    return {
        'PCC_V': pcc_v,
        'PCC_A': pcc_a,
        'RMSE_V': rmse_v,
        'RMSE_A': rmse_a,
        'RMSE_VA': rmse_va_global
    }

### Entraînement des modèles

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_results = {} # Pour stocker les scores finaux

for config in models_dict:
    current_model = config["name"]
    model_type = config["type"]

    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*50}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = config["lr"]
        current_epochs = config["epochs"]
        current_batch_size = config["batch_size"]
        current_dropout = config["dropout"]

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création du Tokenizer et des DataLoaders
        tokenizer = AutoTokenizer.from_pretrained(current_model)
        train_dataset = VADataset(train_df, tokenizer)
        dev_dataset = VADataset(dev_df, tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            print(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = config["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score


ENTRAÎNEMENT DU MODÈLE : SVR_Baseline
Préparation des données pour le SVR...
Vectorisation TF-IDF (max_features=5000)...
Entraînement des modèles SVR (Valence et Arousal)...
Génération des prédictions SVR...

ENTRAÎNEMENT DU MODÈLE : microsoft/deberta-v3-base
Paramètres : LR=1e-05, Epochs=8, Batch=32, Dropout=0.1


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/8 | Train Loss: 4.9933 | Val Loss: 2.1507


Epoch 2/8 | Train Loss: 1.9266 | Val Loss: 2.0829


...

Epoch 4/8 | Train Loss: 1.9171 | Val Loss: 1.4913


Epoch 5/8 | Train Loss: 1.3033 | Val Loss: 1.0898


Epoch 6/8 | Train Loss: 1.0336 | Val Loss: 1.2141


Epoch 7/8 | Train Loss: 0.9212 | Val Loss: 1.1528


Epoch 8/8 | Train Loss: 0.8613 | Val Loss: 1.2182


### Affichage des résultats

In [13]:
print("\n RÉCAPITULATIF DES RÉSULTATS")
for mod, scores in model_results.items():
    print(f"- {mod} : PCC_V = {scores['PCC_V']:.4f} | PCC_A = {scores['PCC_A']:.4f} | RMSE_V = {scores['RMSE_V']:.4f} | RMSE_A = {scores['RMSE_A']:.4f}")


 RÉCAPITULATIF DES RÉSULTATS
- SVR_Baseline : PCC_V = 0.7058 | PCC_A = 0.6137 | RMSE_V = 1.2892 | RMSE_A = 0.8508
- microsoft/deberta-v3-base : PCC_V = 0.8028 | PCC_A = 0.6610 | RMSE_V = 1.2952 | RMSE_A = 0.8830
- bert-base-multilingual-cased : PCC_V = 0.8539 | PCC_A = 0.6947 | RMSE_V = 1.0410 | RMSE_A = 0.8798
- roberta-base : PCC_V = 0.8196 | PCC_A = 0.5839 | RMSE_V = 1.1803 | RMSE_A = 1.0238
